## Find D_max Using Simulated Reliability (For Training)
Takes first occurence of h_dist that did not satisfy reliability threshold FOR ALL LINKS as d_max

In [ ]:
import pandas as pd
import numpy as np 
import math
import os
from tqdm import tqdm
from itertools import product

def get_mcs_index(df_in):
    '''
    Gets the MCS index based on modulation and bitrate column of the df_in
    '''
    df = df_in.copy()
    df["MCS"] = ''
    df.loc[(df["Modulation"] == "BPSK") & (df["Bitrate"] == 6.5), "MCS"] = 0 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 13.0), "MCS"] = 1 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 19.5), "MCS"] = 2 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 26.0), "MCS"] = 3 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 39.0), "MCS"] = 4 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 52.0), "MCS"] = 5 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 58.5), "MCS"] = 6 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 65.0), "MCS"] = 7 # MCS Index 0

    return df

### Compile dataset for finding Dmax

In [ ]:
dl_sim_df_1 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_1_processed/Downlink_Reliability.csv")
ul_sim_df_1 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_1_processed/Uplink_Reliability.csv")
vid_sim_df_1 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_1_processed/Video_Reliability.csv")

dl_sim_df_2 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_2_processed/Downlink_Reliability.csv")
ul_sim_df_2 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_2_processed/Uplink_Reliability.csv")
vid_sim_df_2 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_2_processed/Video_Reliability.csv")

dl_sim_df_3 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/Downlink_Reliability.csv")
ul_sim_df_3 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/Uplink_Reliability.csv")
vid_sim_df_3 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/Video_Reliability.csv")

dl_sim_df_4 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_3_processed/Downlink_Reliability.csv")
ul_sim_df_4 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_3_processed/Uplink_Reliability.csv")
vid_sim_df_4 = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_batch_3_processed/Video_Reliability.csv")
heights = [60, 90, 120, 150, 180, 210, 240, 270, 300]

# For the special case of QAM-16 26 Mbps, 10 ms USI, replace the values in "old" dataset with "newer" results
dl_sim_df_3 = dl_sim_df_3.loc[~((dl_sim_df_3["Bitrate"] == 26) & (dl_sim_df_3["UAV_Sending_Interval"] == 10) & (dl_sim_df_3["Height"].isin(heights)))]
ul_sim_df_3 = ul_sim_df_3.loc[~((ul_sim_df_3["Bitrate"] == 26) & (ul_sim_df_3["UAV_Sending_Interval"] == 10) & (ul_sim_df_3["Height"].isin(heights)))]
vid_sim_df_3 = vid_sim_df_3.loc[~((vid_sim_df_3["Bitrate"] == 26) & (vid_sim_df_3["UAV_Sending_Interval"] == 10) & (vid_sim_df_3["Height"].isin(heights)))]

# # For the special case of QPSK 13 Mbps, 20 ms USI, replace the values in "old" dataset with "newer" results
dl_sim_df_3 = dl_sim_df_3.loc[~((dl_sim_df_3["Bitrate"] == 13) & (dl_sim_df_3["UAV_Sending_Interval"] == 20) & (dl_sim_df_3["Height"].isin(heights)))]
ul_sim_df_3 = ul_sim_df_3.loc[~((ul_sim_df_3["Bitrate"] == 13) & (ul_sim_df_3["UAV_Sending_Interval"] == 20) & (ul_sim_df_3["Height"].isin(heights)))]
vid_sim_df_3 = vid_sim_df_3.loc[~((vid_sim_df_3["Bitrate"] == 13) & (vid_sim_df_3["UAV_Sending_Interval"] == 20) & (vid_sim_df_3["Height"].isin(heights)))]

dl = pd.concat((dl_sim_df_1, dl_sim_df_2, dl_sim_df_3, dl_sim_df_4))
ul = pd.concat((ul_sim_df_1, ul_sim_df_2, ul_sim_df_3, ul_sim_df_4))
vid = pd.concat((vid_sim_df_1, vid_sim_df_2, vid_sim_df_3, vid_sim_df_4))

dl.sort_values(by=["UAV_Sending_Interval", "Bitrate", "Height", "Horizontal_Distance"], inplace=True)
ul.sort_values(by=["UAV_Sending_Interval", "Bitrate", "Height", "Horizontal_Distance"], inplace=True)
vid.sort_values(by=["UAV_Sending_Interval", "Bitrate", "Height", "Horizontal_Distance"], inplace=True)

dl.to_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Downlink_Reliability.csv")
ul.to_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Uplink_Reliability.csv")
vid.to_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Video_Reliability.csv")

### Find Dmax for reliability level

In [ ]:
dl_sim_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Downlink_Reliability.csv")
ul_sim_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Uplink_Reliability.csv")
vid_sim_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/max_hdist_processed/Video_Reliability.csv")

# Get the reliabilities for DL, UL and Video, and record it in dl_sim_df
dl_sim_df["Reliability_DL"] = dl_sim_df["Num_Reliable"] / dl_sim_df["Num_Sent"]
dl_sim_df["Reliability_UL"] = ul_sim_df["Num_Reliable"] / ul_sim_df["Num_Sent"]
dl_sim_df["Reliability_Vid"] = vid_sim_df["Num_Reliable"] / vid_sim_df["Num_Sent"]

dl_sim_df = get_mcs_index(dl_sim_df)

uav_send_int_norm = {10:-1, 20:-0.5, 40:0, 66.7: 0.5, 100:1, 1000:2}
heights = [60, 70, 90, 100, 120, 130, 150, 160, 180, 190, 210, 220, 240, 250, 270, 280, 300]
# horizontal_dist = np.linspace(0, 500, 51, endpoint=True) # Just use the hdist that's available (ASSUMING IT STARTS AT 0)
uav_send_int = [10, 20, 66.7, 100]
mcs_index = np.arange(8).tolist()
reliability_th = 0.999 # Threshold for reliability value
crit_dist_list = []
# for usi, mcs, height in tqdm(list(product(uav_send_int, mcs_index, heights))):
for usi, mcs in tqdm(list(product(uav_send_int, mcs_index))):
    for height in heights:
        crit_distance = 0
        df = dl_sim_df.loc[(dl_sim_df["Height"]==height) & (dl_sim_df["UAV_Sending_Interval"]==usi) & (dl_sim_df["MCS"]==mcs)]
        if (not df.empty):
            hdist_range = df["Horizontal_Distance"].max()
            horizontal_dist = np.arange(0, hdist_range + 10, step=10) # Assumes step size of 10
            for hdist in horizontal_dist:
                df_hdist = df.loc[df["Horizontal_Distance"]==hdist]
                if (df_hdist["Reliability_DL"].values[0] >= reliability_th) & (df_hdist["Reliability_UL"].values[0] >= reliability_th) & (df_hdist["Reliability_Vid"].values[0] >= reliability_th):
                    crit_distance = hdist
                else:
                    break
            if crit_distance == hdist_range:
                print("HDist limit reached. MCS_Index: {}, USI: {}, Height: {}, D_max: {}".format(mcs, usi, height, crit_distance))
        crit_dist_list.append({"MCS_Index": mcs, "USI": usi, "Height": height, "D_max": crit_distance})

crit_dist_df = pd.DataFrame(crit_dist_list)
crit_dist_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/manual_control_max_hdist/Dmax_RelTh999_Simulated_Reliability_v3_10e5.csv", index=False)